# Functions advanced
* in the last lesson we learned the basics of functions (def, parameters, return)
* in this lesson we go deeper: variable amount of arguments, variable scope, lambda functions, recursion, higher-order functions and closures
* these concepts are the base for later lessons (decorators, generators)

## *args and **kwargs - variable amount of arguments
* sometimes we dont know in advance how many arguments a function should accept
* python allows this with a special "*" (args) and "**" (kwargs) syntax

In [1]:
# *args collects any amount of positional arguments into a tuple
def add_all(*numbers):
    print("numbers received as tuple:", numbers, type(numbers))
    total = 0
    for number in numbers:
        total += number
    return total

print(add_all(1, 2, 3))
print(add_all(1, 2, 3, 4, 5))
print(add_all())  # also works with zero arguments, sum stays 0

# **kwargs collects any amount of keyword arguments into a dict
def describe_person(**properties):
    print("properties received as dict:", properties, type(properties))
    for key, value in properties.items():
        print(f"{key}: {value}")

describe_person(name="Alex", age=30, city="Berlin")

# both can be combined with normal parameters and with each other
# order rule: normal params, then *args, then normal keyword params, then **kwargs
def full_example(required_param, *args, default_param="default", **kwargs):
    print(required_param, args, default_param, kwargs)

full_example("mandatory", 1, 2, 3, default_param="custom", extra="info")

numbers received as tuple: (1, 2, 3) <class 'tuple'>
6
numbers received as tuple: (1, 2, 3, 4, 5) <class 'tuple'>
15
numbers received as tuple: () <class 'tuple'>
0
properties received as dict: {'name': 'Alex', 'age': 30, 'city': 'Berlin'} <class 'dict'>
name: Alex
age: 30
city: Berlin
mandatory (1, 2, 3) custom {'extra': 'info'}


In [2]:
# the "*" is also used the other way around to unpack a list/tuple into separate arguments
def add_three(a, b, c):
    return a + b + c

my_values = [1, 2, 3]
print(add_three(*my_values))  # unpacks the list into a=1, b=2, c=3

# the "**" does the same for dicts, matching keys to parameter names
my_kwargs = {"a": 10, "b": 20, "c": 30}
print(add_three(**my_kwargs))

6
60


## variable scope
* scope defines where a variable is visible/accessible
* python has (simplified) two important scopes we need to understand right now: global scope (outside every function) and local scope (inside a function)

In [3]:
# a variable created inside a function is local, it doesnt exist outside the function
def my_function():
    local_variable = "i only exist inside this function"
    print(local_variable)

my_function()
# print(local_variable)  # this would raise a NameError, local_variable is not defined out here

# a function CAN read a global variable without any special keyword
global_variable = "i am global"

def read_global():
    print(global_variable)  # reading works without any keyword

read_global()

# but a function can NOT overwrite a global variable without the "global" keyword
counter = 0

def increment_wrong():
    counter = counter + 1  # this raises an UnboundLocalError! python treats counter as local because of the assignment
    return counter

try:
    increment_wrong()
except UnboundLocalError as error:
    print("error:", error)

def increment_correct():
    global counter  # this tells python: use the global variable, dont create a local one
    counter = counter + 1
    return counter

print(increment_correct())
print(increment_correct())

# expert note: using "global" a lot is normally considered bad style, it makes code harder to reason about
# prefer passing values as parameters and returning results instead of relying on global state

i only exist inside this function
i am global
error: cannot access local variable 'counter' where it is not associated with a value
1
2


## lambda functions
* a lambda is a small, anonymous (nameless), one-expression function
* syntax: lambda parameters: expression
* lambdas are often used where we need a short function "on the fly", for example as an argument to another function

In [4]:
# a normal function
def square(x):
    return x ** 2

# the exact same thing as a lambda
square_lambda = lambda x: x ** 2

print(square(5), square_lambda(5))

# lambdas can take multiple parameters
add = lambda a, b: a + b
print(add(3, 4))

# a very common use case: as the "key" argument for sorting
people = [("Alex", 30), ("Mia", 22), ("Sam", 45)]
people_sorted_by_age = sorted(people, key=lambda person: person[1])
print(people_sorted_by_age)

# expert note: a lambda can only contain a single expression, no statements (no if/for/while blocks, no assignments)
# if your logic needs more than one line, write a normal "def" function instead

25 25
7
[('Mia', 22), ('Alex', 30), ('Sam', 45)]


## higher-order functions: map, filter, functools.reduce
* a higher-order function is a function that accepts another function as an argument (or returns a function)
* python has three classic higher-order functions in its toolbox: map(), filter() and functools.reduce()

In [5]:
from functools import reduce

numbers = [1, 2, 3, 4, 5]

# map() applies a function to every element of an iterable, returns a map-object (itself iterable, so we wrap it into list() to see it)
squared = list(map(lambda n: n ** 2, numbers))
print("map:", squared)

# filter() keeps only the elements where the function returns True
evens = list(filter(lambda n: n % 2 == 0, numbers))
print("filter:", evens)

# reduce() combines all elements into a single value, applying the function step by step (accumulator, current_element)
total = reduce(lambda accumulator, current: accumulator + current, numbers)
print("reduce:", total)

# expert note: today most python developers prefer list comprehensions over map()/filter() because they are more readable
# the comprehension equivalent of the two lines above:
print("comprehension equivalent:", [n ** 2 for n in numbers], [n for n in numbers if n % 2 == 0])
# reduce() however still doesnt have a comprehension equivalent and is the classic tool for this kind of "fold" operation

map: [1, 4, 9, 16, 25]
filter: [2, 4]
reduce: 15
comprehension equivalent: [1, 4, 9, 16, 25] [2, 4]


## closures
* a closure is an inner function that "remembers" variables from its enclosing (outer) function, even after the outer function already finished running
* closures are the base concept behind decorators, which we will learn in a later lesson

In [6]:
def make_multiplier(factor):
    # "factor" belongs to the outer function make_multiplier
    def multiplier(number):
        return number * factor  # the inner function "closes over" factor
    return multiplier  # we return the inner function itself, not the result of calling it

times_three = make_multiplier(3)
times_five = make_multiplier(5)

print(times_three(10))  # 30
print(times_five(10))   # 50
print(times_three(2))   # 6

# even though make_multiplier() already finished executing, "times_three" still remembers factor=3
# this is exactly what a closure is: a function bundled together with its remembered ("closed over") outer variables

# if the inner function needs to modify an outer (but not global) variable, we need the "nonlocal" keyword - the local-scope sibling of "global"
def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

counter = make_counter()
print(counter())
print(counter())
print(counter())

30
50
6
1
2
3


## recursion
* recursion means a function calls itself, usually to solve a problem by breaking it down into smaller versions of the same problem
* every recursive function needs a "base case" (a condition that stops the recursion) - without one you get infinite recursion and eventually a RecursionError

In [7]:
def factorial(n):
    if n <= 1:          # base case, stops the recursion
        return 1
    return n * factorial(n - 1)  # recursive case, calls itself with a smaller problem

print(factorial(5))  # 5 * 4 * 3 * 2 * 1 = 120

def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print([fibonacci(n) for n in range(10)])

# expert note: python has a default recursion limit (usually 1000 nested calls), you can check it with sys.getrecursionlimit()
import sys
print("recursion limit:", sys.getrecursionlimit())
# also note: this naive recursive fibonacci recalculates the same values many times and gets very slow for bigger n
# we will fix this performance problem with functools.lru_cache (memoization/caching) once we learn decorators

120
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
recursion limit: 1000


## quick comparison to other languages
* python's `*args`/`**kwargs` is similar to JavaScript's rest parameters `function f(...args)` or Java's varargs `void f(int... numbers)`, but python additionally separates positional (`*args`) from keyword (`**kwargs`) collection - most other languages dont make that distinction
* python's `lambda x: x*2` corresponds to JavaScript's arrow function `x => x*2`, Java's lambda `x -> x*2`, C#'s `x => x*2`, Rust's closure `|x| x*2` - the concept of a short anonymous function is nearly universal in modern languages
* closures work almost identically in JavaScript, but python needs the explicit `nonlocal` keyword to reassign an outer variable, while JavaScript closures can reassign an outer `let`/`var` without any special keyword
* python's `global`/`nonlocal` requirement is a deliberate design decision: by default an assignment always creates a new local variable, this avoids accidental overwrites that are a common bug source in JavaScript
* recursion works the same everywhere, but python's default recursion limit (~1000) is much lower than languages like Java or C++, where the limit is only bound by available stack memory - very deep recursion in python is a smell that you should rewrite as a loop or increase the limit deliberately with `sys.setrecursionlimit()`

## Exercises
1. write a function `describe(**kwargs)` that prints each received keyword argument as "key: value" and also prints how many keyword arguments were received
2. write a closure `make_power(exponent)` that returns a function which raises its input to the given exponent (`make_power(2)` should behave like a square function, `make_power(3)` like a cube function)
3. write a recursive function `sum_digits(number)` that returns the sum of all digits of a positive integer (e.g. `sum_digits(1234)` returns `10`)
4. use `filter()` and a lambda to get all words longer than 4 characters out of the list `["cat", "python", "hi", "elephant", "sun"]`

In [8]:
# TODO: solve exercise 1 here (describe(**kwargs))


# TODO: solve exercise 2 here (make_power closure)


# TODO: solve exercise 3 here (recursive sum_digits)


# TODO: solve exercise 4 here (filter + lambda for long words)

### sample solutions
* try the exercises above yourself first, only look here if you are stuck or want to compare your approach

In [9]:
# sample solution 1
def describe(**kwargs):
    for key, value in kwargs.items():
        print(f"{key}: {value}")
    print("received", len(kwargs), "keyword arguments")

describe(name="Alex", age=30, city="Berlin")

# sample solution 2
def make_power(exponent):
    def power(base):
        return base ** exponent
    return power

square = make_power(2)
cube = make_power(3)
print(square(4), cube(4))

# sample solution 3
def sum_digits(number):
    if number < 10:
        return number
    return number % 10 + sum_digits(number // 10)

print(sum_digits(1234))

# sample solution 4
words = ["cat", "python", "hi", "elephant", "sun"]
long_words = list(filter(lambda word: len(word) > 4, words))
print(long_words)

name: Alex
age: 30
city: Berlin
received 3 keyword arguments
16 64
10
['python', 'elephant']


* congratulation you finished this lesson

## what we learned
* *args and **kwargs for flexible amounts of arguments, and the reverse "*"/"**" unpacking
* global and local variable scope, the global keyword
* lambda functions and higher-order functions (map, filter, functools.reduce)
* closures and the nonlocal keyword
* recursion, base cases and the python recursion limit